In [ ]:
from routing_fundamentals.qwen_loader import build_embedder, load_generator
from routing_fundamentals.router import load_routes, rank_routes, pick_route
from routing_fundamentals.dispatch import cascade_fallback
from routing_fundamentals.aggregate import semantic_agreement
import yaml

# Load configuration and initialize components
cfg = yaml.safe_load(open("config.yaml"))
routes = load_routes("routes.yaml")

embed_cfg = cfg['embedding']
embed_fn = build_embedder(embed_cfg['model'], normalize=embed_cfg.get('normalize', True))
gen = load_generator(cfg['generator'])

W = cfg['router']['weights']
TOP_K = cfg['router']['top_k']
TEMP  = cfg['router']['temperature']

print("Routes:", [r.name for r in routes], "backend:", getattr(gen, 'backend', type(gen).__name__))

ModuleNotFoundError: No module named 'routing_fundamentals'

# Qwen + Router Playground (package layout)

_Recovered from a corrupted notebook. The cells below re-create imports, configuration, tuning, benchmarking, and simple visualization utilities._

## 1) Interactive Parameter Tuning
Experiment with different routing parameters. Adjust weights and temperature to see how the selected route changes.

In [ ]:
def tune_routing_parameters(query, routes, embed_fn, base_weights, base_temp=0.35, variations=None):
    """Test routing with different parameter combinations.
    Returns a dict keyed by variation name.
    """
    if variations is None:
        variations = {
            'temp_low': {'temp': 0.10, 'weights': base_weights},            'temp_high': {'temp': 0.70, 'weights': base_weights},
            'sim_boost': {'temp': base_temp, 'weights': {**base_weights, 'sim': 2.0}},
            'kw_boost' : {'temp': base_temp, 'weights': {**base_weights, 'kw': 1.0}},
        }

    results = {}
    # Pre-compute the embedding for the query and pass a lambda that returns it
    query_vec = embed_fn([query])[0]

    for var_name, params in variations.items():
        scored = rank_routes(query, routes, lambda x: [query_vec], params['weights'], top_k=min(TOP_K, len(routes)))
        picked = pick_route(scored, tau=params['temp'])
        results[var_name] = {
            'picked_route': getattr(picked, 'name', str(picked)),
            'scores': [(r.name, c['score']) for r, c in scored],
            'params': params
        }

    return results

# Example usage (uncomment to run)
test_queries = [
    "write a python function to sort a list",
    "explain matrix multiplication",
    "find the documentation for API endpoints",
]

print("=== Parameter Tuning Examples ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = tune_routing_parameters(query, routes, embed_fn, W, base_temp=TEMP)
    for var_name, result in results.items():
        score_str = [(n, f"{s:.3f}") for n, s in result['scores']]
        print(f"  {var_name}: {result['picked_route']} (scores: {score_str})")

## 2) Performance Analysis
Measure timing and basic complexity of routing operations.

In [ ]:
import time

def benchmark_routing(routes, embed_fn, weights, queries, num_runs=5):
    """Benchmark routing performance and return a list of timing dicts."""
    results = []
    for query in queries:
        times = []
        for _ in range(num_runs):
            start = time.time()
            query_vec = embed_fn([query])[0]
            scored = rank_routes(query, routes, lambda x: [query_vec], weights, top_k=min(3, len(routes)))
            pick_route(scored, tau=0.35)
            times.append(time.time() - start)
        avg_time = sum(times) / len(times)
        results.append({
            'query': query,
            'avg_time': avg_time,
            'routes_evaluated': len(routes),
            'top_k': min(3, len(routes)),
        })
    return results

# Run performance benchmark (uncomment to re-run)
benchmark_queries = [
    "short query",
    "this is a much longer query about programming and algorithms",
    "write a function",
] * 3  # repeat to stabilize timing

print("\n=== Performance Benchmark ===")
perf_results = benchmark_routing(routes, embed_fn, W, benchmark_queries)
for r in perf_results:
    print(f"{r['avg_time']:.4f}s | top_k={r['top_k']} | routes={r['routes_evaluated']} | '{r['query']}'")
total_time = sum(r['avg_time'] for r in perf_results)
print(f"Total avg time (sum across queries): {total_time:.4f}s")

## 3) Basic Visualization
Simple text-based histogram of routing scores for a given query.

In [ ]:
def visualize_routing(query, routes, embed_fn, weights, temp=0.35):
    """Print a simple text-based bar chart of route scores for a query."""
    query_vec = embed_fn([query])[0]
    scored = rank_routes(query, routes, lambda x: [query_vec], weights, top_k=len(routes))

    print(f"Routing visualization for: '{query}'")
    print("=" * 60)
    max_score = max(c['score'] for _, c in scored)
    min_score = min(c['score'] for _, c in scored)
    score_range = max(max_score - min_score, 1e-9)

    for route, comps in scored:
        normalized = (comps['score'] - min_score) / score_range
        bar_length = int(normalized * 30)
        bar = "█" * bar_length + "░" * (30 - bar_length)
        print(f"{route.name:<24} | score={comps['score']:.3f} | {bar}")

# Example visualization (uncomment to run)
example_query = "write a python function to sort a list"
visualize_routing(example_query, routes, embed_fn, W, temp=TEMP)

## 4) (Optional) Simple Dispatch + Agreement Demo
A tiny sketch showing how one *might* combine dispatch and semantic agreement.

> Note: This is illustrative; adapt to your project's actual generator/route APIs.

In [ ]:
def demo_dispatch_and_agreement(query, routes, embed_fn, weights, top_k=3, temp=0.35):
    # Score and pick top-k candidate routes
    qv = embed_fn([query])[0]
    scored = rank_routes(query, routes, lambda x: [qv], weights, top_k=min(top_k, len(routes)))
    picked = [r for r, _ in scored]

    # Dispatch using a simple cascade fallback (illustrative; adjust to your APIs)
    responses = cascade_fallback(query, picked, gen)

    # Compute semantic agreement across responses (illustrative)
    agreement = semantic_agreement([resp.text for resp in responses], embed_fn)
    print("Picked routes:", [r.name for r in picked])
    for i, resp in enumerate(responses):
        print(f"\nResponse {i+1} from {picked[i].name}:")
        print(resp.text)
    print("\nSemantic agreement score:", agreement)

# Example (commented out to avoid accidental executions)
# demo_dispatch_and_agreement("Explain BFS vs DFS and give a Python snippet", routes, embed_fn, W)